# L08 capstone: your autograd engine against PyTorch

This notebook needs no GPU; any Colab runtime works. Run the cells in order. The last cell is the
autograder: it runs the capstone tests and prints a completion code to paste into the lesson.

You will (1) paste your engine, (2) train a 2-64-3 ReLU network on the spiral with your engine and
with PyTorch from identical initial weights and compare the loss curves, (3) walk a real PyTorch
`grad_fn` graph, and (4) write a custom `torch.autograd.Function` that passes `gradcheck`.

In [ ]:
import os
import numpy as np
import torch

SMALL = os.environ.get("SMALL") == "1"
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print("PyTorch", torch.__version__, "(SMALL mode)" if SMALL else "")

## 1. Your engine

Paste your finished engine from exercise 9 of the lesson (the `Tensor` class with its operations,
`cross_entropy_fused`, `Parameter`, `Linear`, `MLP` and `SGD`) into the next cell, replacing the
`raise` line.

In [ ]:


def unbroadcast(g, shape):
    """Sum g over the axes that broadcasting added or stretched, to get `shape`."""
    while g.ndim > len(shape):
        g = g.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and g.shape[i] != 1:
            g = g.sum(axis=i, keepdims=True)
    return g


def as_tensor(x):
    return x if isinstance(x, Tensor) else Tensor(x)


class Tensor:
    def __init__(self, data, parents=(), op=""):
        self.data = np.array(data, dtype=float)
        self.grad = None
        self._parents = parents
        self._backward = None
        self.op = op

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f"Tensor(shape={self.data.shape}, op={self.op!r})"

    def _accumulate(self, g):
        self.grad = g.copy() if self.grad is None else self.grad + g

    def __add__(self, other):
        other = as_tensor(other)
        out = Tensor(self.data + other.data, (self, other), "add")

        def _backward():
            self._accumulate(unbroadcast(out.grad, self.shape))
            other._accumulate(unbroadcast(out.grad, other.shape))
        out._backward = _backward
        return out

    __radd__ = __add__

    def __mul__(self, other):
        other = as_tensor(other)
        out = Tensor(self.data * other.data, (self, other), "mul")

        def _backward():
            self._accumulate(unbroadcast(out.grad * other.data, self.shape))
            other._accumulate(unbroadcast(out.grad * self.data, other.shape))
        out._backward = _backward
        return out

    __rmul__ = __mul__

    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), (self,), "sum")

        def _backward():
            g = out.grad
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            self._accumulate(np.broadcast_to(g, self.shape).copy())
        out._backward = _backward
        return out

    def backward(self):
        order, seen, stack = [], set(), [(self, False)]
        while stack:
            node, done = stack.pop()
            if done:
                order.append(node)
                continue
            if id(node) in seen:
                continue
            seen.add(id(node))
            stack.append((node, True))
            for p in node._parents:
                if id(p) not in seen:
                    stack.append((p, False))
        for node in order:
            if node._parents:
                node.grad = None  # intermediate gradients do not persist between passes
        self.grad = np.ones_like(self.data)
        for node in reversed(order):
            if node._backward is not None and node.grad is not None:
                node._backward()


def _matmul(self, other):
    other = as_tensor(other)
    out = Tensor(self.data @ other.data, (self, other), "matmul")

    def _backward():
        self._accumulate(out.grad @ other.data.T)
        other._accumulate(self.data.T @ out.grad)
    out._backward = _backward
    return out


def _transpose(self):
    out = Tensor(self.data.T, (self,), "transpose")

    def _backward():
        self._accumulate(out.grad.T)
    out._backward = _backward
    return out


def _reshape(self, *shape):
    out = Tensor(self.data.reshape(*shape), (self,), "reshape")

    def _backward():
        self._accumulate(out.grad.reshape(self.shape))
    out._backward = _backward
    return out


Tensor.__matmul__ = _matmul
Tensor.T = property(_transpose)
Tensor.reshape = _reshape


def _unary(self, value, local, op):
    out = Tensor(value, (self,), op)

    def _backward():
        self._accumulate(out.grad * local())
    out._backward = _backward
    return out


def _exp(self):
    v = np.exp(self.data)
    return _unary(self, v, lambda: v, "exp")


def _log(self):
    return _unary(self, np.log(self.data), lambda: 1.0 / self.data, "log")


def _relu(self):
    return _unary(self, np.maximum(self.data, 0), lambda: (self.data > 0).astype(float), "relu")


def _tanh(self):
    v = np.tanh(self.data)
    return _unary(self, v, lambda: 1.0 - v ** 2, "tanh")


def _pow(self, p):
    return _unary(self, self.data ** p, lambda: p * self.data ** (p - 1), "pow")


def _neg(self):
    return self * -1.0


def _sub(self, other):
    return self + (-as_tensor(other))


def _rsub(self, other):
    return as_tensor(other) + (-self)


def _div(self, other):
    return self * as_tensor(other) ** -1.0


def _rdiv(self, other):
    return as_tensor(other) * self ** -1.0


def _mean(self, axis=None, keepdims=False):
    n = self.data.size if axis is None else self.data.shape[axis]
    return self.sum(axis=axis, keepdims=keepdims) * (1.0 / n)


Tensor.exp = _exp
Tensor.log = _log
Tensor.relu = _relu
Tensor.tanh = _tanh
Tensor.__pow__ = _pow
Tensor.__neg__ = _neg
Tensor.__sub__ = _sub
Tensor.__rsub__ = _rsub
Tensor.__truediv__ = _div
Tensor.__rtruediv__ = _rdiv
Tensor.mean = _mean


def cross_entropy(logits, y):
    """Mean softmax cross-entropy built from primitive operations."""
    m = Tensor(logits.data.max(axis=1, keepdims=True))  # constant: the loss does not depend on it
    z = logits - m
    lse = z.exp().sum(axis=1, keepdims=True).log()
    onehot = np.eye(logits.shape[1])[y]
    picked = (z * onehot).sum(axis=1, keepdims=True)
    return (lse - picked).mean()


def cross_entropy_fused(logits, y):
    """The same loss as one operation with the backward rule (P - onehot) / B."""
    Z = logits.data - logits.data.max(axis=1, keepdims=True)
    logp = Z - np.log(np.exp(Z).sum(axis=1, keepdims=True))
    B = Z.shape[0]
    out = Tensor(-np.mean(logp[np.arange(B), y]), (logits,), "cross_entropy")

    def _backward():
        P = np.exp(logp)
        P[np.arange(B), y] -= 1.0
        logits._accumulate(out.grad * P / B)
    out._backward = _backward
    return out


class Module:
    def parameters(self):
        params = []
        for v in self.__dict__.values():
            if isinstance(v, Tensor) and getattr(v, "is_param", False):
                params.append(v)
            elif isinstance(v, Module):
                params.extend(v.parameters())
            elif isinstance(v, (list, tuple)):
                for item in v:
                    if isinstance(item, Module):
                        params.extend(item.parameters())
        return params

    def zero_grad(self):
        for p in self.parameters():
            p.grad = None

    def __call__(self, x):
        return self.forward(x)


def Parameter(data):
    t = Tensor(data)
    t.is_param = True
    return t


class Linear(Module):
    def __init__(self, n_in, n_out, rng):
        self.W = Parameter(rng.normal(size=(n_in, n_out)) * np.sqrt(2.0 / n_in))
        self.b = Parameter(np.zeros(n_out))

    def forward(self, x):
        return x @ self.W + self.b


class MLP(Module):
    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.layers = [Linear(a, b, rng) for a, b in zip(sizes, sizes[1:])]

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = x.relu()
        return x


class SGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


## 2. Data and the engine run

In [ ]:
def make_spiral(n=100, K=3, seed=0):
    rng = np.random.default_rng(seed)
    Xs, ys = [], []
    for j in range(K):
        r = np.linspace(0.0, 1.0, n)
        t = np.linspace(4 * j, 4 * (j + 1), n) + 0.2 * rng.normal(size=n)
        Xs.append(np.column_stack([r * np.sin(t), r * np.cos(t)]))
        ys.append(np.full(n, j))
    return np.concatenate(Xs), np.concatenate(ys)


X, y = make_spiral()
SIZES = [2, 64, 3]
STEPS = 40 if SMALL else 300
LR = 1.0

model = MLP(SIZES, seed=0)
init_weights = [(layer.W.data.copy(), layer.b.data.copy()) for layer in model.layers]
opt = SGD(model.parameters(), LR)
engine_losses = []
for _ in range(STEPS):
    opt.zero_grad()
    loss = cross_entropy_fused(model(Tensor(X)), y)
    loss.backward()
    opt.step()
    engine_losses.append(float(loss.data))
print("engine: first loss", round(engine_losses[0], 4), "last loss", round(engine_losses[-1], 4))

## 3. The same network in PyTorch

Build `TorchMLP` from the engine's initial weights. Note that `torch.nn.Linear` stores its weight as
`(out_features, in_features)`, the transpose of your engine's `W`. Then train it with
`torch.optim.SGD` and `torch.nn.functional.cross_entropy` for the same number of steps.

In [ ]:
class TorchMLP(torch.nn.Module):
    def __init__(self, init):
        super().__init__()
        self.layers = torch.nn.ModuleList()
        for W, b in init:
            lin = torch.nn.Linear(W.shape[0], W.shape[1])
            with torch.no_grad():
                lin.weight.copy_(torch.tensor(W.T))
                lin.bias.copy_(torch.tensor(b))
            self.layers.append(lin)

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = torch.relu(x)
        return x


tmodel = TorchMLP(init_weights)
topt = torch.optim.SGD(tmodel.parameters(), lr=LR)
Xt, yt = torch.tensor(X), torch.tensor(y)
torch_losses = []
for _ in range(STEPS):
    topt.zero_grad()
    tloss = torch.nn.functional.cross_entropy(tmodel(Xt), yt)
    tloss.backward()
    topt.step()
    torch_losses.append(tloss.item())
print("largest difference between the loss curves:", max(abs(a - b) for a, b in zip(engine_losses, torch_losses)))

## 4. Walk the graph PyTorch recorded

Every non-leaf tensor has a `grad_fn`; its `next_functions` are `(function, index)` pairs leading to
the functions that produced its inputs. Write `walk_grad_fn(t)` returning the class names of every
function reachable from `t.grad_fn` (leaf nodes appear as `AccumulateGrad`; `None` entries mark
inputs that need no gradient).

In [ ]:
def walk_grad_fn(t):
    names, seen, stack = [], set(), [t.grad_fn]
    while stack:
        fn = stack.pop()
        if fn is None or fn in seen:
            continue
        seen.add(fn)
        names.append(type(fn).__name__)
        for nxt, _ in fn.next_functions:
            stack.append(nxt)
    return names


grad_fn_names = walk_grad_fn(torch.nn.functional.cross_entropy(tmodel(Xt), yt))
print(sorted(set(grad_fn_names)))

## 5. A custom autograd function

Implement the squared ReLU, $f(x) = \max(0, x)^2$, as a `torch.autograd.Function` with your own
backward rule, then verify it with `torch.autograd.gradcheck` and against PyTorch's automatic
derivative of the same expression.

In [ ]:
class SquaredReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.clamp(x, min=0) ** 2

    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * 2 * torch.clamp(x, min=0)


xc = torch.randn(25, requires_grad=True)
custom_gradcheck_ok = bool(torch.autograd.gradcheck(SquaredReLU.apply, (xc,)))
(g_custom,) = torch.autograd.grad(SquaredReLU.apply(xc).sum(), xc)
(g_auto,) = torch.autograd.grad((torch.relu(xc) ** 2).sum(), xc)
custom_matches_autograd = float((g_custom - g_auto).abs().max())
print("gradcheck:", custom_gradcheck_ok, "| max difference from autograd:", custom_matches_autograd)

In [ ]:
# Forge autograder: run this cell last. It runs the capstone tests and prints your completion code.
import hashlib as _hashlib, sys as _sys, types as _types, unittest as _unittest
_FORGE_ID = "L08"
_FORGE_TESTS = '''import numpy as np
import sub


def test_engine_matches_pytorch():
    e = np.array(sub.engine_losses)
    t = np.array(sub.torch_losses)
    assert len(e) == len(t) >= 20
    assert np.max(np.abs(e - t)) < 1e-6, np.max(np.abs(e - t))


def test_engine_learns():
    assert sub.engine_losses[-1] < 0.7 * sub.engine_losses[0]


def test_grad_fn_walk():
    ops = set(sub.grad_fn_names)
    assert any("Addmm" in o or "Mm" in o for o in ops), ops
    assert any("Relu" in o for o in ops), ops
    assert any("NllLoss" in o or "LogSoftmax" in o for o in ops), ops
    assert "AccumulateGrad" in ops


def test_custom_function():
    assert sub.custom_gradcheck_ok is True
    assert sub.custom_matches_autograd < 1e-12
'''
sub = _types.SimpleNamespace(**{_k: _v for _k, _v in globals().items() if not _k.startswith('_')})
_sys.modules['sub'] = sub
_ns = {'__name__': 'forge_tests'}
exec(compile(_FORGE_TESTS, 'tests.py', 'exec'), _ns)
_passed = []
for _name, _fn in list(_ns.items()):
    if _name.startswith('test_') and callable(_fn):
        try:
            _fn()
            _passed.append(_name)
            print('PASS', _name)
        except _unittest.SkipTest as _e:
            print('SKIP', _name, '-', _e)
        except Exception as _e:
            print('FAIL', _name, '-', type(_e).__name__, _e)
_digest = _hashlib.sha256(f"{_FORGE_ID}:{','.join(sorted(_passed))}:forge".encode()).hexdigest()[:8]
print(f"Completion code: FORGE-{_FORGE_ID}-{_digest}")